In [132]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold,cross_val_score, train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder,StandardScaler,OrdinalEncoder

from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR



In [133]:
pd.set_option('display.max_columns', None)

In [134]:
original_df = pd.read_csv('4.6_dataset_feature_selected.csv')
original_df

,sector,price,super_area,bedrooms,bathroom,balcony,age_possession,servant_room,luxury_category,parking,building_type
0,jharsa,0.70,1278.0000,2,2,2,1 to 5 year old,0,medium,1,low-rise
1,sohna,0.38,682.3526,2,2,2,1 to 5 year old,0,medium,2,mid-rise
2,sector 71,1.35,1198.0000,2,2,2,10+ year old,0,high,1,mid-rise
3,sector 82,3.40,3240.0000,4,4,2,0 to 1 year old,1,medium,2,low-rise
4,sohna,0.95,1065.0000,2,2,2,0 to 1 year old,0,low,1,low-rise
...,...,...,...,...,...,...,...,...,...,...,...
5587,sohna,0.88,1313.5000,4,4,2,5 to 10 year old,0,medium,2,mid-rise
5588,sector 99,3.75,3500.0000,4,4,3+,1 to 5 year old,1,high,2,mid-rise
5589,sector 92,1.45,1628.0000,3,4,3+,1 to 5 year old,1,medium,1,mid-rise
5590,sector 65,3.85,2916.6800,3,3,2,0 to 1 year old,0,medium,0,high-rise


In [135]:
df = original_df.copy()
df

,sector,price,super_area,bedrooms,bathroom,balcony,age_possession,servant_room,luxury_category,parking,building_type
0,jharsa,0.70,1278.0000,2,2,2,1 to 5 year old,0,medium,1,low-rise
1,sohna,0.38,682.3526,2,2,2,1 to 5 year old,0,medium,2,mid-rise
2,sector 71,1.35,1198.0000,2,2,2,10+ year old,0,high,1,mid-rise
3,sector 82,3.40,3240.0000,4,4,2,0 to 1 year old,1,medium,2,low-rise
4,sohna,0.95,1065.0000,2,2,2,0 to 1 year old,0,low,1,low-rise
...,...,...,...,...,...,...,...,...,...,...,...
5587,sohna,0.88,1313.5000,4,4,2,5 to 10 year old,0,medium,2,mid-rise
5588,sector 99,3.75,3500.0000,4,4,3+,1 to 5 year old,1,high,2,mid-rise
5589,sector 92,1.45,1628.0000,3,4,3+,1 to 5 year old,1,medium,1,mid-rise
5590,sector 65,3.85,2916.6800,3,3,2,0 to 1 year old,0,medium,0,high-rise


In [136]:
X = df.drop(columns = ['price'])
Y= df['price']

In [137]:
y_transformed = np.log1p(Y)

#### Applying column tranformations and building pipeline

In [138]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5592 entries, 0 to 5591
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   sector           5592 non-null   object 
 1   super_area       5592 non-null   float64
 2   bedrooms         5592 non-null   int64  
 3   bathroom         5592 non-null   int64  
 4   balcony          5592 non-null   object 
 5   age_possession   5592 non-null   object 
 6   servant_room     5592 non-null   int64  
 7   luxury_category  5592 non-null   object 
 8   parking          5592 non-null   int64  
 9   building_type    5592 non-null   object 
dtypes: float64(1), int64(4), object(5)
memory usage: 437.0+ KB


In [139]:
#creating column transformer for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ordinal_order = [
    ['0', '1', '2', '3', '3+'],
    ['under construction','0 to 1 year old','1 to 5 year old', '5 to 10 year old','10+ year old' ],
    ['low','medium','high'],
]

ordinal_cols = [4,5,7]

preprocessor = ColumnTransformer(
    transformers= [
        ('scaling', StandardScaler(), ['super_area','bedrooms','bathroom','servant_room','parking']),
        ('ordinal', OrdinalEncoder(categories=ordinal_order),ordinal_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore',drop='first'),['sector','building_type'])
				],
    remainder='passthrough'
)


In [140]:
#building pipeline
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold,cross_val_score,train_test_split

def scorer (model_name,model):
	output =[]
	output.append(model_name)
	pipeline = Pipeline(
		[('preprocessor',preprocessor),
			('model',model)
			]
	)

	#K-fold cross validation
	kfold = KFold(n_splits = 10, shuffle= True)
	scores = cross_val_score(pipeline,X,y_transformed, cv= kfold , scoring='r2')

	output.append(scores.mean())

	#train test split
	x_train,x_test,y_train,y_test = train_test_split(X,y_transformed,test_size=0.2)

	pipeline.fit(x_train,y_train)

	y_pred = pipeline.predict(x_test)

	y_pred = np.expm1(y_pred)

	output.append(mean_absolute_error(np.expm1(y_test),y_pred))

	return output


In [141]:
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.neural_network import MLPRegressor

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [142]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

In [143]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [144]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.937542,0.281722
10,xgboost,0.931854,0.336632
5,random forest,0.924783,0.342647
9,mlp,0.931319,0.362709
4,decision tree,0.883170,0.392043
1,svr,0.921141,0.426404
0,linear_reg,0.875345,0.491782
2,ridge,0.883608,0.493243
7,gradient boosting,0.870988,0.502010
8,adaboost,0.716225,0.787886


In [165]:
## Extra trees pipeline
pipeline_1 = Pipeline(
		[('preprocessor',preprocessor),
			('model',ExtraTreesRegressor())
			]
	)

k_fold = KFold(n_splits =10 , shuffle=True)
scores_1 = cross_val_score(estimator=pipeline_1, X=X, y=y_transformed, cv= k_fold , scoring= 'r2' )

x_train,x_test,y_train,y_test = train_test_split(X,y_transformed,test_size=0.2)
pipeline_1.fit(x_train,y_train)
y_pred = pipeline_1.predict(x_test)
y_pred = np.expm1(y_pred)

mae_1 = mean_absolute_error(np.expm1(y_test),y_pred)
mae_1

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

0.3116430348707561

In [167]:
scores_1.mean(),scores_1.std(), mae_1

(0.9367213964199774, 0.007270412788880864, 0.3116430348707561)

### Making an ensemble of models

In [168]:
from sklearn.ensemble import VotingRegressor

etr = ExtraTreesRegressor(n_estimators=100,)
rfr = RandomForestRegressor(n_estimators=100)
xgb = XGBRegressor(n_estimators=100)

ensemble = VotingRegressor([
    ('etr', etr),
    ('rfr', rfr),
    ('xgb', xgb)
])


pipeline_2 = Pipeline([('preprocessor',preprocessor),
					 ('model',ensemble)]
	)

#K-fold cross validation
kfold = KFold(n_splits = 10, shuffle= True)
scores_2 = cross_val_score(pipeline_2,X,y_transformed, cv= kfold , scoring='r2')


#train test split
x_train,x_test,y_train,y_test = train_test_split(X,y_transformed,test_size=0.2)

pipeline_2.fit(x_train,y_train)

y_pred = pipeline_2.predict(x_test)

y_pred = np.expm1(y_pred)

mae_2 = mean_absolute_error(np.expm1(y_test),y_pred)
mae_2

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

0.3190789206997326

In [169]:
scores_2.mean(), scores_2.std(),mae_2

(0.9397300455693014, 0.010100709731172983, 0.3190789206997326)

-- No improvement over ExtraTrees pipeline

In [ ]:
#testing on real world data
#https://www.99acres.com/3-bhk-bedroom-apartment-flat-for-sale-in-krisumi-waterfall-residences-sector-36a-gurgaon-2538-sq-ft-spid-Z81368679
data = [['sector 36a',2538,3,3,'3+','0 to 1 year old',1,'high',2,'high-rise']]
cols = X.columns

testing_df= pd.DataFrame(data,columns=cols)
testing_df = testing_df.astype(X.dtypes.to_dict())

np.expm1(pipeline.predict(testing_df))


data = [['sector 110a',1407,3,3,'2','1 to 5 year old',0,'medium',1,'mid-rise']]
cols = X.columns

testing_df= pd.DataFrame(data,columns=cols)
testing_df = testing_df.astype(X.dtypes.to_dict())

np.expm1(pipeline.predict(testing_df))
np.expm1(pipeline_2.predict(testing_df))

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


array([1.82712422])

### Hyper parameter tuning

In [170]:
#creating column transformer for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ordinal_order = [
    ['0', '1', '2', '3', '3+'],
    ['under construction','0 to 1 year old','1 to 5 year old', '5 to 10 year old','10+ year old' ],
    ['low','medium','high'],
]

ordinal_cols = [4,5,7]

preprocessor = ColumnTransformer(
    transformers= [
        ('scaling', StandardScaler(), ['super_area','bedrooms','bathroom','servant_room','parking']),
        ('ordinal', OrdinalEncoder(categories=ordinal_order),ordinal_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore',drop='first'),['sector','building_type'])
				],
    remainder='passthrough'
)


In [171]:
#build pipeline
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['auto', 'sqrt'],
    'regressor__bootstrap': [True]
}

pipeline_3 = Pipeline(
    [('preprocessor',preprocessor),
     ('regressor',ExtraTreesRegressor())])

kfold = KFold(n_splits=10, shuffle=True, random_state=42)
search = GridSearchCV(pipeline_3, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)
search.fit(X,y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits
[CV 4/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=auto, regressor__max_samples=0.1, regressor__n_estimators=50;, score=nan total time=   0.0s
[CV 6/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=auto, regressor__max_samples=0.1, regressor__n_estimators=50;, score=nan total time=   0.0s
[CV 3/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=auto, regressor__max_samples=0.1, regressor__n_estimators=50;, score=nan total time=   0.0s
[CV 2/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=auto, regressor__max_samples=0.1, regressor__n_estimators=50;, score=nan total time=   0.0s
[CV 7/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=auto, regressor__max_samples=0.1, regressor__n_estimators=50;, score=nan total time=   0.0s
[CV 1/10] END r

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV 1/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.844 total time=   0.2s
[CV 9/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.858 total time=   0.2s


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categorie

[CV 2/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.833 total time=   0.2s
[CV 1/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.25, regressor__n_estimators=50;, score=0.895 total time=   0.4s
[CV 10/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.1, regressor__n_estimators=50;, score=0.855 total time=   0.2s
[CV 9/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.25, regressor__n_estimators=50;, score=0.902 total time=   0.4s
[CV 7/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=0.1, regressor__n_estimators=100;, score=0.862 total time=   0.5s


...

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


[CV 9/10] END regressor__bootstrap=True, regressor__max_depth=30, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.910 total time=   4.8s
[CV 10/10] END regressor__bootstrap=True, regressor__max_depth=30, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.906 total time=   4.8s


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:540: FitFailedWarning: 
640 fits failed out of a total of 1280.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
194 fits failed with the following error:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 888, in _fit_and_score
    estimator.fit(X_

[CV 10/10] END regressor__bootstrap=True, regressor__max_depth=None, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.923 total time=   5.4s


GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('scaling',
                                                                         StandardScaler(),
                                                                         ['super_area',
                                                                          'bedrooms',
                                                                          'bathroom',
                                                                          'servant_room',
                                                                          'parking']),
                                                                        ('ordinal',
                                                                         OrdinalEncoder(categories=[['0',
                                                                                                     '1',
                                                                                                     '2',
                                                                                                     '3',
                                                                                                     '3+'],
                                                                                                    ['under '
                                                                                                     'construction',
                                                                                                     '0 '
                                                                                                     'to '
                                                                                                     '1 '
                                                                                                     'ye...
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         ['sector',
                                                                          'building_type'])])),
                                       ('regressor', ExtraTreesRegressor())]),
             n_jobs=-1,
             param_grid={'regressor__bootstrap': [True],
                         'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__max_features': ['auto', 'sqrt'],
                         'regressor__max_samples': [0.1, 0.25, 0.5, 1.0],
                         'regressor__n_estimators': [50, 100, 200, 300]},
             scoring='r2', verbose=4)

In [172]:
pipeline_3 = search.best_estimator_

In [173]:
search.best_params_

{'regressor__bootstrap': True,
 'regressor__max_depth': None,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [174]:
search.best_score_

0.9248101834919948

In [175]:
x_train,x_test,y_train,y_test = train_test_split(X,y_transformed,test_size=0.2)

pipeline_3.fit(x_train,y_train)

y_pred = pipeline_3.predict(x_test)

y_pred = np.expm1(y_pred)

mae_3 = mean_absolute_error(np.expm1(y_test),y_pred)
mae_3

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:242: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


0.3773416327098663

In [176]:
search.best_score_,mae_3

(0.9248101834919948, 0.3773416327098663)

-- Somehow, performing worse

In [177]:
model_data = [['ExtraTreesRegressor',scores_1.mean(), scores_1.std(),mae_1],
              ['Ensemble models',scores_2.mean(), scores_2.std(),mae_2],
              ['ExtraTrees Parameter tunes',search.best_score_, np.nan,mae_3]]
pd.DataFrame(model_data, columns = ['Model name','scores_mean', 'scores_std','mae'])

,Model name,scores_mean,scores_std,mae
0,ExtraTreesRegressor,0.936721,0.007270,0.311643
1,Ensemble models,0.939730,0.010101,0.319079
2,ExtraTrees Parameter tunes,0.924810,NaN,0.377342


-- Based on this choosing pipeline 1

# Exporting final model and dataset

In [179]:
#creating column transformer for preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ordinal_order = [
    ['0', '1', '2', '3', '3+'],
    ['under construction','0 to 1 year old','1 to 5 year old', '5 to 10 year old','10+ year old' ],
    ['low','medium','high'],
]

ordinal_cols = [4,5,7]

preprocessor = ColumnTransformer(
    transformers= [
        ('scaling', StandardScaler(), ['super_area','bedrooms','bathroom','servant_room','parking']),
        ('ordinal', OrdinalEncoder(categories=ordinal_order),ordinal_cols),
        ('onehot', OneHotEncoder(handle_unknown='ignore',drop='first'),['sector','building_type'])
				],
    remainder='passthrough'
)

final_pipe =  Pipeline(
		[('preprocessor',preprocessor),
			('model',ExtraTreesRegressor())
			]
	)

final_pipe.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('scaling', StandardScaler(),
                                                  ['super_area', 'bedrooms',
                                                   'bathroom', 'servant_room',
                                                   'parking']),
                                                 ('ordinal',
                                                  OrdinalEncoder(categories=[['0',
                                                                              '1',
                                                                              '2',
                                                                              '3',
                                                                              '3+'],
                                                                             ['under '
                                                                              'construction',
                                                                              '0 '
                                                                              'to '
                                                                              '1 '
                                                                              'year '
                                                                              'old',
                                                                              '1 '
                                                                              'to '
                                                                              '5 '
                                                                              'year '
                                                                              'old',
                                                                              '5 '
                                                                              'to '
                                                                              '10 '
                                                                              'year '
                                                                              'old',
                                                                              '10+ '
                                                                              'year '
                                                                              'old'],
                                                                             ['low',
                                                                              'medium',
                                                                              'high']]),
                                                  [4, 5, 7]),
                                                 ('onehot',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['sector',
                                                   'building_type'])])),
                ('model', ExtraTreesRegressor())])

In [180]:
#export pipeline
import pickle

with open('5.2_piepline.pkl', 'wb') as file:
    pickle.dump(pipeline_1,file)

In [181]:
#export dataset
with open('5.3_dataset_final.pkl','wb') as file:
    pickle.dump(X,file)